# LLM Multi-Seed Aggregation — Qwen2.5-7B-Instruct (5 seeds across 2 batches)

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Environment:** Kaggle CPU (no GPU required), Internet not required
**Estimated runtime:** ~5 minutes (CPU only)

---

## Purpose

This CPU-only notebook aggregates results from **NB9a** (seeds 42, 123, 2024) and **NB9b** (seeds 7, 99) into the final 5-seed mean±std summary for **Qwen2.5-7B-Instruct** on the Bengali Yellow Journalism Detection task. It addresses the methodological gap left by NB5, which reported only a single-seed F1=0.075 with seed=42.

**No GPU required. No model loading, no fine-tuning. Just CSV aggregation + plotting.**

## Why a separate aggregation notebook?

The full 5-seed run takes ~13 hours on a Kaggle T4 GPU, exceeding Kaggle's 12-hour commit limit. We therefore split it into two GPU notebooks:

| Notebook | Seeds | Runtime | Output CSV |
|----------|-------|---------|------------|
| NB9a (Batch 1) | [42, 123, 2024] | ~7.5 h | `multi_seed_qwen7b_batch1_results.csv` |
| NB9b (Batch 2) | [7, 99]         | ~5 h   | `multi_seed_qwen7b_batch2_results.csv` |

This notebook (NB9c) loads both batch CSVs and produces the final 5-seed summary.

## Inputs

**Add both batch result CSVs as Kaggle input datasets before running.**

1. **Batch 1 dataset** (e.g., named `swarabyanjan-qwen7b-batch1`) containing `multi_seed_qwen7b_batch1_results.csv` — produced by NB9a.
2. **Batch 2 dataset** (e.g., named `swarabyanjan-qwen7b-batch2`) containing `multi_seed_qwen7b_batch2_results.csv` — produced by NB9b.

The notebook auto-discovers the CSVs via `glob` over `/kaggle/input/**`.

## Outputs

- `multi_seed_qwen7b_final_summary.json` — canonical 5-seed aggregate summary (F1/Acc/Prec/Rec/Kappa/MCC mean±std, per-seed results, batch provenance)
- `multi_seed_f1_all_seeds.png` — bar plot of F1 per seed with mean / ±1σ / NB5 single-seed reference lines


## How to Provide Batch Result CSVs as Inputs

This notebook needs TWO CSV files produced by NB9a and NB9b:

| File | Produced by | Expected seeds |
|---|---|---|
| `multi_seed_qwen7b_batch1_results.csv` | NB9a_Qwen-7B_seeds_42_123_2024 | [42, 123, 2024] |
| `multi_seed_qwen7b_batch2_results.csv` | NB9b_Qwen-7B_seeds_7_99 | [7, 99] |

### Method 1 (Recommended): Attach notebook outputs directly

On Kaggle:

1. Run `NB9a_Qwen-7B_seeds_42_123_2024.ipynb` → Save Version → wait for completion (~7.5h)
2. Run `NB9b_Qwen-7B_seeds_7_99.ipynb` → Save Version → wait for completion (~5h)
3. Open `NB9c_Qwen-7B_aggregate.ipynb` for editing
4. In the right panel → **"Add Input"** → **"Notebook Output"**
5. Search for your NB9a notebook → click **+**
6. Search for your NB9b notebook → click **+**
7. Both notebooks' output files will appear under `/kaggle/input/<notebook-slug>/`
8. **Save Version → Save & Run All** (CPU only, ~5 minutes)

The notebook auto-discovers the CSVs via recursive glob — you don't need to know the exact slug.

### Method 2 (Alternative): Upload as Kaggle datasets

1. Download `multi_seed_qwen7b_batch1_results.csv` from NB9a's output panel
2. Create a new Kaggle dataset (e.g., `swarabyanjan-qwen7b-batch1`) and upload the CSV
3. Repeat for NB9b's output → `swarabyanjan-qwen7b-batch2`
4. In NB9c → "Add Input" → "Dataset" → add both datasets
5. The CSVs will appear under `/kaggle/input/<dataset-slug>/`

Both methods work — the notebook's file discovery handles either.


In [1]:
# === Kaggle Input Diagnostic ===
# This cell helps you verify that the batch result CSVs are accessible.
# Run this cell first; if it fails, follow the instructions in the error message.

import os
import glob

print('='*70)
print('KAGGLE INPUT DIAGNOSTIC')
print('='*70)

if os.path.exists('/kaggle/input/'):
    print(f'\n✅ Running on Kaggle. Contents of /kaggle/input/:')
    input_dirs = sorted(os.listdir('/kaggle/input/'))
    if not input_dirs:
        print('  ⚠️  /kaggle/input/ is EMPTY. You need to add inputs:')
        print('     - Right panel → "Add Input" → "Notebook Output" → select NB9a')
        print('     - Repeat for NB9b')
    else:
        for d in input_dirs:
            dpath = f'/kaggle/input/{d}'
            if os.path.isdir(dpath):
                files = sorted(os.listdir(dpath))
                csv_files = [f for f in files if f.endswith('.csv')]
                json_files = [f for f in files if f.endswith('.json')]
                png_files = [f for f in files if f.endswith('.png')]
                print(f'\n  📁 {d}/')
                if csv_files:
                    print(f'     CSV files ({len(csv_files)}):')
                    for f in csv_files:
                        size_mb = os.path.getsize(f'{dpath}/{f}') / 1e6
                        print(f'       📄 {f} ({size_mb:.2f} MB)')
                if json_files:
                    print(f'     JSON files ({len(json_files)}):')
                    for f in json_files[:5]:
                        print(f'       📄 {f}')
                if png_files:
                    print(f'     PNG files ({len(png_files)}):')
                    for f in png_files[:5]:
                        print(f'       📄 {f}')
                if not (csv_files or json_files or png_files):
                    print(f'     (no CSV/JSON/PNG files; first 5 items: {files[:5]})')
            else:
                print(f'  📄 {d} (file)')
else:
    print(f'\n💻 Not running on Kaggle. Checking local paths...')

# Check for the specific batch CSVs
print(f'\n{"="*70}')
print('SEARCHING FOR BATCH RESULT CSVs')
print('='*70)

batch1_found = glob.glob('/kaggle/input/**/multi_seed_qwen7b_batch1_results.csv', recursive=True)
batch2_found = glob.glob('/kaggle/input/**/multi_seed_qwen7b_batch2_results.csv', recursive=True)

# Also check local paths
for local_path in ['./', '../results/', './results/',
                    '/home/z/my-project/analysis/github_repo/results/',
                    '/home/z/my-project/']:
    batch1_found += glob.glob(f'{local_path}multi_seed_qwen7b_batch1_results.csv')
    batch2_found += glob.glob(f'{local_path}multi_seed_qwen7b_batch2_results.csv')

print(f'\nBatch 1 CSV (multi_seed_qwen7b_batch1_results.csv):')
if batch1_found:
    for m in batch1_found:
        print(f'  ✅ {m}')
else:
    print(f'  ❌ NOT FOUND')

print(f'\nBatch 2 CSV (multi_seed_qwen7b_batch2_results.csv):')
if batch2_found:
    for m in batch2_found:
        print(f'  ✅ {m}')
else:
    print(f'  ❌ NOT FOUND')

if not (batch1_found and batch2_found):
    print(f'\n⚠️  One or both batch CSVs are missing. See instructions in the')
    print(f'   markdown cell above for how to attach notebook outputs as inputs.')
    print(f'\n   If you have NOT yet run NB9a and NB9b, do that first:')
    print(f'   - NB9a (~7.5h on T4 GPU): produces batch1 CSV')
    print(f'   - NB9b (~5h on T4 GPU): produces batch2 CSV')
else:
    print(f'\n✅ Both batch CSVs found. You can proceed to run all cells.')


KAGGLE INPUT DIAGNOSTIC

✅ Running on Kaggle. Contents of /kaggle/input/:

  📁 notebooks/
     (no CSV/JSON/PNG files; first 5 items: ['swagotammalakar'])

SEARCHING FOR BATCH RESULT CSVs

Batch 1 CSV (multi_seed_qwen7b_batch1_results.csv):
  ✅ /kaggle/input/notebooks/swagotammalakar/nb9a-qwen-7b-seeds-42-123-2024/multi_seed_qwen7b_batch1_results.csv

Batch 2 CSV (multi_seed_qwen7b_batch2_results.csv):
  ✅ /kaggle/input/notebooks/swagotammalakar/nb9b-qwen-7b-seeds-7-99/multi_seed_qwen7b_batch2_results.csv

✅ Both batch CSVs found. You can proceed to run all cells.


### 1. Environment Setup

Minimal imports only. **No `torch`, no `transformers`, no `peft`, no `trl`, no `bitsandbytes`, no `!pip install`.** This notebook is CPU-only and uses packages pre-installed in the Kaggle Python environment.


In [2]:
# Standard scientific stack — all pre-installed in Kaggle's Python 3 environment.
import os
import glob
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless backend (Kaggle commit mode)
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn is available but only needed if recomputing metrics — we trust the
# values already computed in NB9a/NB9b, so we don't import from sklearn.metrics here.
# If you need to recompute, uncomment the next line:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')

# Font that handles basic characters and minus signs
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f"pandas {pd.__version__}, numpy {np.__version__}", flush=True)
print(f"matplotlib {matplotlib.__version__}, seaborn {sns.__version__}", flush=True)
print(f"NB9c: CPU-only aggregation of NB9a + NB9b multi-seed results", flush=True)


pandas 2.3.3, numpy 2.0.2
matplotlib 3.10.0, seaborn 0.13.2
NB9c: CPU-only aggregation of NB9a + NB9b multi-seed results


### 2. Configuration — Auto-Discover Batch Result CSVs

The two batch result CSVs are located by globbing under `/kaggle/input/**` (the standard Kaggle input mount point). Multiple search patterns are tried in order of specificity. The search also falls back to `./` and `../results/` for local testing outside Kaggle.


In [3]:
# === Configuration ===

# Expected batch result CSV filenames (produced by NB9a and NB9b)
BATCH1_CSV_FILENAME = 'multi_seed_qwen7b_batch1_results.csv'  # from NB9a (seeds 42, 123, 2024)
BATCH2_CSV_FILENAME = 'multi_seed_qwen7b_batch2_results.csv'  # from NB9b (seeds 7, 99)

# Output paths
OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
FINAL_SUMMARY_FILE = f'{OUTPUT_DIR}/multi_seed_qwen7b_final_summary.json'
FINAL_PLOT_FILE = f'{OUTPUT_DIR}/multi_seed_f1_all_seeds.png'

# Reference: NB5 single-seed F1 (seed=42) for comparison
NB5_F1 = 0.075


def find_batch_csv(filename, label='Batch'):
    """
    Find a batch result CSV file. Searches:
    1. /kaggle/input/ recursively (notebook outputs AND datasets)
    2. Local paths (./, ../results/, /home/z/my-project/...)

    On Kaggle, when you attach another notebook's OUTPUT as an input to this
    notebook, the files appear under /kaggle/input/<source-notebook-slug>/.
    The slug is auto-generated from the notebook title and is unpredictable,
    so we use recursive glob discovery.
    """
    print(f'[{label}] Searching for: {filename}')

    # 1. Kaggle input paths — recursive glob handles any subdirectory structure
    #    (covers both "notebook output as input" and "dataset as input")
    kaggle_matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)

    # 2. Local paths (for testing outside Kaggle)
    local_candidates = [
        f'./{filename}',
        f'../results/{filename}',
        f'./results/{filename}',
        f'/home/z/my-project/analysis/github_repo/results/{filename}',
        f'/home/z/my-project/{filename}',
    ]
    local_matches = [p for p in local_candidates if os.path.isfile(p)]

    all_matches = kaggle_matches + local_matches

    if not all_matches:
        # No match found — print diagnostic info and raise helpful error
        print(f'\n[{label}] ❌ FILE NOT FOUND: {filename}')
        print(f'[{label}] Searched:')
        print(f'  - /kaggle/input/**/{filename} (recursive)')
        for p in local_candidates:
            print(f'  - {p}')

        # List what IS in /kaggle/input/ to help debug
        if os.path.exists('/kaggle/input/'):
            print(f'\n[{label}] Contents of /kaggle/input/:')
            for entry in sorted(os.listdir('/kaggle/input/')):
                entry_path = f'/kaggle/input/{entry}'
                if os.path.isdir(entry_path):
                    files = os.listdir(entry_path)
                    csv_files = [f for f in files if f.endswith('.csv')]
                    print(f'  📁 {entry}/')
                    if csv_files:
                        for f in csv_files[:10]:
                            print(f'      📄 {f}')
                    else:
                        print(f'      (no CSV files; contents: {files[:5]})')
                else:
                    print(f'  📄 {entry}')

        raise FileNotFoundError(
            f'\n{"="*70}\n'
            f'COULD NOT FIND: {filename}\n\n'
            f'To fix this on Kaggle:\n'
            f'1. Make sure you have run NB9a (and NB9b for batch2) and that\n'
            f'   the notebook completed successfully (check the output panel\n'
            f'   for multi_seed_qwen7b_batch1_results.csv).\n\n'
            f'2. In NB9c\'s right panel → "Add Input" → "Notebook Output" →\n'
            f'   search for your NB9a notebook → click the + button.\n'
            f'   Repeat for NB9b.\n\n'
            f'3. The output files will appear under\n'
            f'   /kaggle/input/<notebook-slug>/multi_seed_qwen7b_batch1_results.csv\n'
            f'   and this notebook will auto-discover them.\n\n'
            f'Alternative: download the CSV from NB9a/NB9b, upload it as a\n'
            f'Kaggle dataset, and attach that dataset as input instead.\n'
            f'{"="*70}'
        )

    # Found at least one match — pick the first, warn if multiple
    chosen = all_matches[0]
    print(f'[{label}] ✅ Found: {chosen}')
    if len(all_matches) > 1:
        print(f'[{label}] ⚠️  Multiple matches found (using first):')
        for m in all_matches:
            marker = '→' if m == chosen else ' '
            print(f'  {marker} {m}')

    return chosen


# Discover batch CSVs (handles both notebook-output-as-input AND dataset-as-input)
print('Discovering batch result CSVs...\n')
BATCH1_CSV = find_batch_csv(BATCH1_CSV_FILENAME, label='Batch 1')
print()
BATCH2_CSV = find_batch_csv(BATCH2_CSV_FILENAME, label='Batch 2')

print(f'\n{"="*70}')
print('CONFIGURATION SUMMARY')
print('='*70)
print(f'Batch 1 CSV: {BATCH1_CSV}')
print(f'Batch 2 CSV: {BATCH2_CSV}')
print(f'Output dir:  {OUTPUT_DIR}')
print(f'Summary:     {FINAL_SUMMARY_FILE}')
print(f'Plot:        {FINAL_PLOT_FILE}')
print(f'NB5 ref F1:  {NB5_F1}')


Discovering batch result CSVs...

[Batch 1] Searching for: multi_seed_qwen7b_batch1_results.csv
[Batch 1] ✅ Found: /kaggle/input/notebooks/swagotammalakar/nb9a-qwen-7b-seeds-42-123-2024/multi_seed_qwen7b_batch1_results.csv

[Batch 2] Searching for: multi_seed_qwen7b_batch2_results.csv
[Batch 2] ✅ Found: /kaggle/input/notebooks/swagotammalakar/nb9b-qwen-7b-seeds-7-99/multi_seed_qwen7b_batch2_results.csv

CONFIGURATION SUMMARY
Batch 1 CSV: /kaggle/input/notebooks/swagotammalakar/nb9a-qwen-7b-seeds-42-123-2024/multi_seed_qwen7b_batch1_results.csv
Batch 2 CSV: /kaggle/input/notebooks/swagotammalakar/nb9b-qwen-7b-seeds-7-99/multi_seed_qwen7b_batch2_results.csv
Output dir:  /kaggle/working
Summary:     /kaggle/working/multi_seed_qwen7b_final_summary.json
Plot:        /kaggle/working/multi_seed_f1_all_seeds.png
NB5 ref F1:  0.075


### 3. Load and Combine Batch Results

Both batch CSVs are loaded into pandas DataFrames and concatenated. Each CSV has one row per seed (with columns `seed`, `Accuracy`, `Precision`, `Recall`, `F1`, `Kappa`, `MCC`, and possibly `error` for failed runs).


In [4]:
# === Load and Combine Batch Results ===
print(f'Loading batch CSVs...')

batch1_df = pd.read_csv(BATCH1_CSV)
batch2_df = pd.read_csv(BATCH2_CSV)

print(f'\nBatch 1 ({BATCH1_CSV}):')
print(f'  Shape: {batch1_df.shape}')
print(f'  Columns: {list(batch1_df.columns)}')
if 'seed' in batch1_df.columns:
    print(f'  Seeds: {batch1_df["seed"].tolist()}')

print(f'\nBatch 2 ({BATCH2_CSV}):')
print(f'  Shape: {batch2_df.shape}')
print(f'  Columns: {list(batch2_df.columns)}')
if 'seed' in batch2_df.columns:
    print(f'  Seeds: {batch2_df["seed"].tolist()}')

# Validate expected seeds
expected_batch1_seeds = {42, 123, 2024}
expected_batch2_seeds = {7, 99}
if 'seed' in batch1_df.columns:
    actual_batch1_seeds = set(batch1_df['seed'].tolist())
    if actual_batch1_seeds != expected_batch1_seeds:
        print(f'\n⚠️  Batch 1 seed mismatch! Expected {expected_batch1_seeds}, got {actual_batch1_seeds}')
if 'seed' in batch2_df.columns:
    actual_batch2_seeds = set(batch2_df['seed'].tolist())
    if actual_batch2_seeds != expected_batch2_seeds:
        print(f'\n⚠️  Batch 2 seed mismatch! Expected {expected_batch2_seeds}, got {actual_batch2_seeds}')

# Check for duplicate seeds across batches
all_seeds = []
if 'seed' in batch1_df.columns:
    all_seeds.extend(batch1_df['seed'].tolist())
if 'seed' in batch2_df.columns:
    all_seeds.extend(batch2_df['seed'].tolist())
if len(all_seeds) != len(set(all_seeds)):
    print(f'\n⚠️  Duplicate seeds detected across batches: {all_seeds}')
    # Keep only first occurrence of each seed
    seen = set()
    keep_mask = []
    for s in all_seeds:
        if s not in seen:
            seen.add(s)
            keep_mask.append(True)
        else:
            keep_mask.append(False)
    print(f'   Keeping first occurrence of each seed.')

# Tag each row with its batch for traceability (used by filter + comparison cells)
batch1_df['batch'] = 1
batch2_df['batch'] = 2

# Combine
all_results_df = pd.concat([batch1_df, batch2_df], ignore_index=True)
print(f'\n✅ Combined: {len(all_results_df)} seed runs')
print(all_results_df)


Loading batch CSVs...

Batch 1 (/kaggle/input/notebooks/swagotammalakar/nb9a-qwen-7b-seeds-42-123-2024/multi_seed_qwen7b_batch1_results.csv):
  Shape: (3, 21)
  Columns: ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Kappa', 'MCC', 'TP', 'FP', 'FN', 'TN', 'Unparseable', 'Parseable_Pct', 'Train_Min', 'Eval_Min', 'Source', 'Total_Min', 'MAX_SEQ_LEN', 'BATCH_SIZE', 'seed', 'error']
  Seeds: [42, 123, 2024]

Batch 2 (/kaggle/input/notebooks/swagotammalakar/nb9b-qwen-7b-seeds-7-99/multi_seed_qwen7b_batch2_results.csv):
  Shape: (2, 21)
  Columns: ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Kappa', 'MCC', 'TP', 'FP', 'FN', 'TN', 'Unparseable', 'Parseable_Pct', 'Train_Min', 'Eval_Min', 'Source', 'Total_Min', 'MAX_SEQ_LEN', 'BATCH_SIZE', 'seed', 'error']
  Seeds: [7, 99]

✅ Combined: 5 seed runs
                 Model  Accuracy  Precision  Recall      F1  Kappa     MCC  \
0  Qwen2.5-7B-Instruct    0.5130     1.0000   0.026  0.0506  0.026  0.1147   
1                  NaN       NaN

### 4. Filter Successful Runs

Each batch records failures as rows with an `error` column populated (NaN for successful runs). Filter to keep only successful runs for aggregate statistics. Failed runs (if any) are reported so the user knows which seeds need re-running.


In [5]:
# ============================================================
# FILTER SUCCESSFUL RUNS
# ============================================================

if 'error' in all_results_df.columns:
    successful = all_results_df[all_results_df['error'].isna()].reset_index(drop=True).copy()
    failed = all_results_df[~all_results_df['error'].isna()].reset_index(drop=True).copy()
    print(f"Successful: {len(successful)} / {len(all_results_df)}", flush=True)
    if len(failed) > 0:
        print(f"\nFailed runs:", flush=True)
        print(failed[['seed', 'batch', 'error']].to_string(index=False), flush=True)
        print(f"\nNOTE: Failed seeds are EXCLUDED from aggregate statistics.", flush=True)
        print(f"      Re-run the corresponding batch notebook to recover them.", flush=True)
else:
    successful = all_results_df.copy().reset_index(drop=True)
    print(f"All {len(successful)} runs successful (no 'error' column present).", flush=True)

# Coerce metric columns to numeric (CSV round-trip can turn them into strings)
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'Kappa', 'MCC']
for c in metric_cols:
    if c in successful.columns:
        successful[c] = pd.to_numeric(successful[c], errors='coerce')

# Sort by seed for cleaner display
if 'seed' in successful.columns:
    successful = successful.sort_values('seed').reset_index(drop=True)

print(f"\nSuccessful runs (sorted by seed):", flush=True)
cols_to_show = [c for c in ['seed', 'batch', 'Accuracy', 'Precision', 'Recall', 'F1', 'Kappa', 'MCC'] if c in successful.columns]
print(successful[cols_to_show].to_string(index=False), flush=True)


Successful: 3 / 5

Failed runs:
 seed  batch                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      error
  123      1  CUDA out of memory. Tried to allocate 2.03 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.93 GiB is free. Including non-PyTorch memory, this process has 12.63 GiB memory in use. Of the allocated memory 12.40 GiB is allocated by PyTorch, and 92.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to

### 5. Aggregate Statistics

Compute mean / std / min / max across all 5 seeds for each metric. The primary numbers of interest are **F1 mean±std** and **Accuracy mean±std**, which directly address the methodological gap of single-seed reporting.


In [6]:
# ============================================================
# AGGREGATE STATISTICS — all 5 seeds
# ============================================================

metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'Kappa', 'MCC']
metrics_cols = [c for c in metrics_cols if c in successful.columns]

agg = successful[metrics_cols].agg(['mean', 'std', 'min', 'max'])

print('\n' + '=' * 70, flush=True)
print('  Multi-Seed Aggregated Results (Qwen2.5-7B-Instruct, 5 seeds)', flush=True)
print('=' * 70, flush=True)
print(agg.round(4).to_string(), flush=True)
print('=' * 70, flush=True)

print(f'\nF1:        {successful["F1"].mean():.4f} ± {successful["F1"].std():.4f}', flush=True)
print(f'Accuracy:  {successful["Accuracy"].mean():.4f} ± {successful["Accuracy"].std():.4f}', flush=True)
print(f'Precision: {successful["Precision"].mean():.4f} ± {successful["Precision"].std():.4f}', flush=True)
print(f'Recall:    {successful["Recall"].mean():.4f} ± {successful["Recall"].std():.4f}', flush=True)
print(f'Kappa:     {successful["Kappa"].mean():.4f} ± {successful["Kappa"].std():.4f}', flush=True)
print(f'MCC:       {successful["MCC"].mean():.4f} ± {successful["MCC"].std():.4f}', flush=True)



  Multi-Seed Aggregated Results (Qwen2.5-7B-Instruct, 5 seeds)
      Accuracy  Precision  Recall      F1   Kappa     MCC
mean    0.5108     0.8889   0.026  0.0504  0.0217  0.0921
std     0.0038     0.1924   0.000  0.0003  0.0075  0.0391
min     0.5065     0.6667   0.026  0.0500  0.0130  0.0470
max     0.5130     1.0000   0.026  0.0506  0.0260  0.1147

F1:        0.0504 ± 0.0003
Accuracy:  0.5108 ± 0.0038
Precision: 0.8889 ± 0.1924
Recall:    0.0260 ± 0.0000
Kappa:     0.0217 ± 0.0075
MCC:       0.0921 ± 0.0391


### 6. Visualization — Bar Plot of F1 per Seed

Bar plot of F1 per seed (across both batches), with horizontal reference lines for the multi-seed mean, ±1σ, and the NB5 single-seed F1=0.075. Seed 42 is highlighted in green because it matches the NB5 evaluation seed.


In [7]:
# ============================================================
# VISUALIZATION — F1 per seed (all 5 seeds across both batches)
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

seeds = successful['seed'].astype(str)
f1s = successful['F1'].values
mean_f1 = successful['F1'].mean()
std_f1 = successful['F1'].std()

# green for seed 42 (matches NB5); blue for the others
colors = ['#2196F3' if s != '42' else '#4CAF50' for s in seeds]
bars = ax.bar(seeds, f1s, color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)

ax.axhline(mean_f1, color='red', linestyle='--', linewidth=1.5,
           label=f'Mean = {mean_f1:.4f}')
ax.axhline(mean_f1 + std_f1, color='orange', linestyle=':', linewidth=1,
           label=f'+1σ = {mean_f1+std_f1:.4f}')
ax.axhline(mean_f1 - std_f1, color='orange', linestyle=':', linewidth=1,
           label=f'−1σ = {mean_f1-std_f1:.4f}')
ax.axhline(0.075, color='purple', linestyle='-.', linewidth=1,
           label='NB5 single-seed F1 = 0.075')

for bar, f1 in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{f1:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Random Seed')
ax.set_ylabel('F1 Score')
ax.set_title('Qwen2.5-7B-Instruct QLoRA F1 across 5 Random Seeds\n'
             '(Bengali Yellow Journalism Detection)')
ax.set_ylim(0, max(0.15, max(f1s) * 1.3))
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.savefig(FINAL_PLOT_FILE, dpi=150)
print(f'Saved plot to {FINAL_PLOT_FILE}', flush=True)
plt.show()


Saved plot to /kaggle/working/multi_seed_f1_all_seeds.png


### 7. Comparison with NB5 Single-Seed Result

The original NB5 reported F1 = 0.075 for Qwen2.5-7B-Instruct with `seed=42`. This cell compares that single-seed number to the 5-seed mean±std and reports whether NB5's value falls within 1 (or 2) standard deviations of the multi-seed mean — the criterion reviewers asked us to check.


In [8]:
# ============================================================
# COMPARE — NB5 single-seed vs NB9 5-seed multi-seed
# ============================================================

nb5_f1 = NB5_F1  # 0.075, from NB5 (single seed=42)
multi_seed_mean = successful['F1'].mean()
multi_seed_std = successful['F1'].std()

print(f'NB5 single-seed F1 (seed=42):        {nb5_f1:.4f}', flush=True)
print(f'NB9 multi-seed F1 mean (5 seeds):    {multi_seed_mean:.4f} ± {multi_seed_std:.4f}', flush=True)
print(f'Difference:                          {multi_seed_mean - nb5_f1:+.4f}', flush=True)

within_1std = abs(multi_seed_mean - nb5_f1) <= multi_seed_std
within_2std = abs(multi_seed_mean - nb5_f1) <= 2 * multi_seed_std

print(f'\nNB5 result within 1 std of multi-seed mean: {within_1std}', flush=True)
print(f'NB5 result within 2 std of multi-seed mean: {within_2std}', flush=True)

if within_1std:
    print('\n→ NB5 single-seed result is consistent with the multi-seed distribution.', flush=True)
    print('  The single-seed evaluation in NB5 was not an outlier.', flush=True)
else:
    print('\n→ NB5 single-seed result deviates from the multi-seed mean by > 1 std.', flush=True)
    print('  This suggests run-to-run variance is significant and should be reported.', flush=True)

# Per-seed breakdown vs NB5
print(f"\nPer-seed F1 vs NB5 single-seed F1={nb5_f1:.4f}:", flush=True)
for _, row in successful.iterrows():
    diff = float(row['F1']) - nb5_f1
    print(f"  seed={int(row['seed']):<5}  batch={int(row['batch'])}  "
          f"F1={float(row['F1']):.4f}  (Δ={diff:+.4f})", flush=True)


NB5 single-seed F1 (seed=42):        0.0750
NB9 multi-seed F1 mean (5 seeds):    0.0504 ± 0.0003
Difference:                          -0.0246

NB5 result within 1 std of multi-seed mean: False
NB5 result within 2 std of multi-seed mean: False

→ NB5 single-seed result deviates from the multi-seed mean by > 1 std.
  This suggests run-to-run variance is significant and should be reported.

Per-seed F1 vs NB5 single-seed F1=0.0750:
  seed=7      batch=2  F1=0.0506  (Δ=-0.0244)
  seed=42     batch=1  F1=0.0506  (Δ=-0.0244)
  seed=2024   batch=1  F1=0.0500  (Δ=-0.0250)


### 8. Save Final Aggregated Summary

Writes a JSON summary to `multi_seed_qwen7b_final_summary.json` containing the model name, the 5-seed aggregate mean±std for each metric, per-seed results, batch provenance, and the NB5 comparison result. This file is the canonical artefact for inclusion in the paper's supplementary materials.


In [9]:
# ============================================================
# SAVE FINAL AGGREGATED SUMMARY JSON
# ============================================================

final_summary = {
    "model": "Qwen2.5-7B-Instruct",
    "n_seeds_total": len(successful),
    "seeds": successful['seed'].tolist(),
    "batches": [
        {"batch": 1, "seeds": [42, 123, 2024], "csv": "multi_seed_qwen7b_batch1_results.csv"},
        {"batch": 2, "seeds": [7, 99], "csv": "multi_seed_qwen7b_batch2_results.csv"},
    ],
    "f1_mean": float(successful['F1'].mean()),
    "f1_std": float(successful['F1'].std()),
    "f1_min": float(successful['F1'].min()),
    "f1_max": float(successful['F1'].max()),
    "accuracy_mean": float(successful['Accuracy'].mean()),
    "accuracy_std": float(successful['Accuracy'].std()),
    "precision_mean": float(successful['Precision'].mean()),
    "precision_std": float(successful['Precision'].std()) if 'Precision' in successful.columns else None,
    "recall_mean": float(successful['Recall'].mean()),
    "recall_std": float(successful['Recall'].std()) if 'Recall' in successful.columns else None,
    "kappa_mean": float(successful['Kappa'].mean()),
    "kappa_std": float(successful['Kappa'].std()) if 'Kappa' in successful.columns else None,
    "mcc_mean": float(successful['MCC'].mean()),
    "mcc_std": float(successful['MCC'].std()) if 'MCC' in successful.columns else None,
    "per_seed_results": successful.to_dict(orient='records'),
    "nb5_single_seed_f1": 0.075,
    "nb5_within_1std_of_multiseed": bool(within_1std),
    "note": ("Aggregated from 2 batch runs (NB9a + NB9b) due to Kaggle's "
             "12-hour session limit. Addresses reviewer concern about "
             "single-seed LLM evaluation in NB5."),
}

with open(FINAL_SUMMARY_FILE, 'w') as f:
    json.dump(final_summary, f, indent=2, default=str)
print(f'Saved final summary to {FINAL_SUMMARY_FILE}', flush=True)
print(flush=True)
print(json.dumps(final_summary, indent=2, default=str), flush=True)


Saved final summary to /kaggle/working/multi_seed_qwen7b_final_summary.json

{
  "model": "Qwen2.5-7B-Instruct",
  "n_seeds_total": 3,
  "seeds": [
    7,
    42,
    2024
  ],
  "batches": [
    {
      "batch": 1,
      "seeds": [
        42,
        123,
        2024
      ],
      "csv": "multi_seed_qwen7b_batch1_results.csv"
    },
    {
      "batch": 2,
      "seeds": [
        7,
        99
      ],
      "csv": "multi_seed_qwen7b_batch2_results.csv"
    }
  ],
  "f1_mean": 0.0504,
  "f1_std": 0.0003464101615137734,
  "f1_min": 0.05,
  "f1_max": 0.0506,
  "accuracy_mean": 0.5108333333333334,
  "accuracy_std": 0.003752776749732603,
  "precision_mean": 0.8889,
  "precision_std": 0.1924308447209023,
  "recall_mean": 0.026,
  "recall_std": 0.0,
  "kappa_mean": 0.021666666666666667,
  "kappa_std": 0.007505553499465134,
  "mcc_mean": 0.09213333333333333,
  "mcc_std": 0.03908661322413766,
  "per_seed_results": [
    {
      "Model": "Qwen2.5-7B-Instruct",
      "Accuracy": 0.513,
    

### 9. Next Steps

After this notebook completes:

1. **Download outputs from the Kaggle output panel:**
   - `multi_seed_qwen7b_final_summary.json` — canonical 5-seed summary
   - `multi_seed_f1_all_seeds.png` — bar plot of F1 per seed

2. **Copy them to `results/` in the GitHub repo:**
   ```bash
   cp multi_seed_qwen7b_final_summary.json /path/to/repo/results/
   cp multi_seed_f1_all_seeds.png /path/to/repo/outputs/
   ```

3. **Update `results/README.md`** and the top-level `README.md` Results Summary table to include a new row:
   ```
   Qwen2.5-7B-Instruct (QLoRA, 5-seed mean) | {f1_mean} ± {f1_std} | {acc_mean} | LLM Fine-tuned (multi-seed)
   ```
   Substitute the actual `f1_mean`, `f1_std`, and `acc_mean` values from `multi_seed_qwen7b_final_summary.json`.

4. **Note in the paper** that this multi-seed result replaces (or supplements) the single-seed Qwen-7B row from NB5. The original single-seed F1 = 0.075 (seed=42) is preserved as a reference value inside the summary JSON (`nb5_single_seed_f1` field).

5. **If you want to extend** this multi-seed analysis to the other 5 LLMs (Gemma-2-2B-it, Qwen2.5-3B-Instruct, Phi-3-mini-4k, Llama-3.1-8B-Instruct, Gemma-2-9B-it): duplicate NB9a/NB9b with a different `MODEL_CONFIG`, run them on Kaggle, then duplicate this aggregation notebook (changing the CSV filenames) to combine the new batches.
